# AEVB MNIST Reproduction v3

This notebook runs a paper-aligned practical reproduction of Kingma and Welling's Auto-Encoding Variational Bayes experiments on MNIST. The implementation lives in `src/aevb_v3`; this notebook is intentionally a thin, reproducible Colab driver.

## Executed Notebook Note

This notebook preserves the Colab outputs from the completed v3 run. Personal Colab execution metadata and the verbose final zip output were removed before committing. The clean no-output notebook remains available for rerunning the experiment from scratch.


## Scope

- Main dataset: statically binarized MNIST with seed `2026`.
- Main method: AEVB/VAE with one hidden layer, `tanh`, Gaussian encoder, Bernoulli decoder, Adagrad, and explicit MAP L2 penalty.
- Baseline: Wake-Sleep with the same architecture and optimizer.
- Low-dimensional experiment: `z_dim=3`, `hidden_dim=100`, HMC marginal likelihood estimate and MCEM baseline.
- Reliability: metrics are flushed during training, intermediate checkpoints support resume, and the suite runner backs up results after each stage.
- Evaluation scope follows the original paper rather than adding modern diagnostics such as active units or IWAE estimates.

In [ ]:
from pathlib import Path
import os
import shutil

PROJECT_NAME_CANDIDATES = [
    'AEVB_MNIST_Reproduction_v3',
    'AEVB_MNIST_Reproduction_v3_Submission',
]
LOCAL_PROJECT_ROOT = Path('/content') / 'AEVB_MNIST_Reproduction_v3'
DRIVE_OUTPUT_DIR = None

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def is_project_root(path: Path) -> bool:
    return (
        (path / 'pyproject.toml').exists()
        and (path / 'scripts').exists()
        and (path / 'src' / 'aevb_v3').exists()
    )

def find_drive_project_root():
    drive_root = Path('/content/drive/MyDrive')
    candidates = []
    for project_name in PROJECT_NAME_CANDIDATES:
        candidates.extend([
            drive_root / 'ASI' / 'Assignment' / project_name,
            drive_root / project_name,
        ])
    for candidate in candidates:
        if is_project_root(candidate):
            return candidate

    matches = [p.parent for p in drive_root.rglob('pyproject.toml') if is_project_root(p.parent)]
    matches = [p for p in matches if 'aevb' in p.name.lower()] or matches
    if not matches:
        raise FileNotFoundError(
            'Could not find the AEVB v3 project in Google Drive. Upload the whole project folder, not only this notebook.'
        )
    return matches[0]

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_PROJECT_ROOT = find_drive_project_root()
    DRIVE_OUTPUT_DIR = DRIVE_PROJECT_ROOT.parent
    LOCAL_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    for name in ['configs', 'scripts', 'src', 'notebooks']:
        src = DRIVE_PROJECT_ROOT / name
        dst = LOCAL_PROJECT_ROOT / name
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    for name in ['README.md', 'requirements.txt', 'pyproject.toml', '.gitattributes', '.gitignore']:
        shutil.copy2(DRIVE_PROJECT_ROOT / name, LOCAL_PROJECT_ROOT / name)
    os.chdir(LOCAL_PROJECT_ROOT)
else:
    if not (Path.cwd() / 'pyproject.toml').exists():
        raise RuntimeError('Run this notebook from the repository root.')

PROJECT_ROOT = Path.cwd()
print('Project root:', PROJECT_ROOT)
print('Drive project root:', DRIVE_PROJECT_ROOT if IN_COLAB else PROJECT_ROOT)
print('Drive output directory:', DRIVE_OUTPUT_DIR)


Mounted at /content/drive
Project root: /content/AEVB_MNIST_Reproduction_v3
Drive output directory: /content/drive/MyDrive


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for aevb-mnist-reproduction-v3 (pyproject.toml) ... done


In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


## Recommended Suite Run

This is the recommended entrypoint. It runs smoke tests, the learning-rate pilot, AEVB, Wake-Sleep, marginal likelihood, figures, result collection, and a Drive backup after every stage.

In [ ]:
!python scripts/run_v3_suite.py --backup-dir /content/drive/MyDrive/AEVB_MNIST_Reproduction_v3_backups


$ /usr/bin/python3 scripts/run_smoke_tests.py --config configs/quick_debug.yaml
100% 9.91M/9.91M [00:01<00:00, 5.63MB/s]
100% 28.9k/28.9k [00:00<00:00, 132kB/s]
100% 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100% 4.54k/4.54k [00:00<00:00, 16.2MB/s]
aevb_static_bernoulli_z10_h32_seed0_lr0p01_e3_smoke: 100% 3/3 [00:00<00:00,  3.45it/s, test_nelbo=275.77]
Smoke tests passed. Initial mini-batch NELBO: 543.31

$ /usr/bin/python3 scripts/run_pilot.py --config configs/aevb_binarized_practical.yaml
aevb_static_bernoulli_z10_h500_seed0_lr0p01_e5_pilot: 100% 5/5 [00:13<00:00,  2.79s/it, test_nelbo=138.84]
aevb_static_bernoulli_z10_h500_seed0_lr0p02_e5_pilot: 100% 5/5 [00:13<00:00,  2.79s/it, test_nelbo=147.11]
aevb_static_bernoulli_z10_h500_seed0_lr0p1_e5_pilot: 100% 5/5 [00:14<00:00,  2.80s/it, test_nelbo=12224.61]
Selected Adagrad learning rate: 0.01
Wrote selected lr manifest: /content/AEVB_MNIST_Reproduction_v3/results/manifests/selected_lr.json

$ /usr/bin/python3 scripts/backup_results.py --res

## Emergency Backup

Run this cell whenever you want to archive the current `results/` folder manually.

In [ ]:
!python scripts/backup_results.py --results-dir results --output-dir /content/drive/MyDrive/AEVB_MNIST_Reproduction_v3_backups --name AEVB_MNIST_Reproduction_v3_emergency

## Experiment 1: AEVB Lower-Bound Sweep

This runs `z_dim=[3,5,10,20,200]` with three seeds plus `z_dim=2` with seed 0 for the latent manifold figure. The full configuration uses 300 epochs.

In [ ]:
# Optional manual command; skip this if you used the suite runner above.
# !python scripts/run_aevb_sweep.py --config configs/aevb_binarized_practical.yaml --resume

## Experiment 2: Wake-Sleep Baseline

Wake-Sleep is trained with the same MLP architecture, initialization, batch size, optimizer, and latent dimensions as AEVB, using seed 0 for the baseline comparison. The same ELBO evaluator is used for comparison.

In [ ]:
# Optional manual command; skip this if you used the suite runner above.
# !python scripts/run_wake_sleep.py --config configs/wake_sleep_binarized_practical.yaml --resume

## Experiment 3: Low-Dimensional Marginal Likelihood

This experiment follows the paper's low-dimensional setting with a focused configuration: `z_dim=3`, `hidden_dim=100`, `N_train=1000`, AEVB, Wake-Sleep, and MCEM. It estimates marginal likelihood with HMC posterior samples and a full-covariance Gaussian density estimator.

In [ ]:
# Optional manual command; skip this if you used the suite runner above.
# !python scripts/run_marginal_likelihood.py --config configs/marginal_likelihood_z3_h100_practical.yaml --resume

## Figures and Result Tables

In [ ]:
!python scripts/make_figures.py --config configs/aevb_binarized_practical.yaml
!python scripts/collect_results.py --results-dir results

Figures written to: /content/AEVB_MNIST_Reproduction_v3/results/figures
Wrote tables: ['results/tables/final_summary_by_seed.csv', 'results/tables/final_test_summary_by_method_z.csv']
Validation: {'csv_count': 29, 'issues': []}
Wrote manifest: results/manifests/experiment_manifest.json


In [ ]:
from pathlib import Path
for subdir in ['metrics', 'tables', 'figures', 'manifests']:
    files = sorted((Path('results') / subdir).glob('*'))
    print(f'{subdir}: {len(files)} files')
    for path in files[:10]:
        print(' -', path)

metrics: 29 files
 - results/metrics/aevb_static_bernoulli_z10_h500_seed0_lr0p01_e300.csv
 - results/metrics/aevb_static_bernoulli_z10_h500_seed0_lr0p01_e5_pilot.csv
 - results/metrics/aevb_static_bernoulli_z10_h500_seed0_lr0p02_e5_pilot.csv
 - results/metrics/aevb_static_bernoulli_z10_h500_seed0_lr0p1_e5_pilot.csv
 - results/metrics/aevb_static_bernoulli_z10_h500_seed1_lr0p01_e300.csv
 - results/metrics/aevb_static_bernoulli_z10_h500_seed2_lr0p01_e300.csv
 - results/metrics/aevb_static_bernoulli_z200_h500_seed0_lr0p01_e300.csv
 - results/metrics/aevb_static_bernoulli_z200_h500_seed1_lr0p01_e300.csv
 - results/metrics/aevb_static_bernoulli_z200_h500_seed2_lr0p01_e300.csv
 - results/metrics/aevb_static_bernoulli_z20_h500_seed0_lr0p01_e300.csv
tables: 2 files
 - results/tables/final_summary_by_seed.csv
 - results/tables/final_test_summary_by_method_z.csv
figures: 8 files
 - results/figures/aevb_binarized_random_samples_z10_seed0.png
 - results/figures/aevb_binarized_random_samples_z200_s

## Export Results

The result folder should be committed or archived with the code. Checkpoints can be stored with Git LFS or uploaded as a GitHub Release asset.

In [ ]:
!zip -r AEVB_MNIST_Reproduction_v3_results.zip results README.md configs scripts src pyproject.toml requirements.txt .gitattributes .gitignore

if DRIVE_OUTPUT_DIR is not None:
    destination = Path(DRIVE_OUTPUT_DIR) / 'AEVB_MNIST_Reproduction_v3_results.zip'
    shutil.copy2('AEVB_MNIST_Reproduction_v3_results.zip', destination)
    print('Copied archive to:', destination)